
# Capítulo 5: Imitation — Generación de Audio Sintético y Clave

📖 `Libro: Investigación en Ciencia de Datos`

**Autor: Dora María Ballesteros Larrotta**  
Universidad Militar Nueva Granada (UMNG)

---

## Material complementario del libro

Este notebook acompaña el **Capítulo 5** y contiene un ejemplo de generación de una señal de audio sintética y su correspondiente clave mediante un proceso de **imitación (Imitation)**.

El procedimiento permite obtener dos elementos: un **audio sintético** y una **clave**, que posteriormente se utilizan en el proceso inverso (**Reverse**) para recuperar la señal de audio original.

---


In [1]:
# ============================================================
# 🔐 IMITATION: GENERACIÓN DE CINCO CLAVES SECRETAS
# ============================================================
# Algoritmo original: Dora Maria Ballesteros
# Actualizado:  21/10/2026
#
# 🎵 Entrada:
#    audio.wav       → Target común
#    secret1.wav     → Mensaje secreto 1
#    secret2.wav     → Mensaje secreto 2
#    secret3.wav     → Mensaje secreto 3
#    secret4.wav     → Mensaje secreto 4
#    secret5.wav     → Mensaje secreto 5
#
# 🔑 Salida:
#    key1.csv ... key5.csv
#    imitated1.wav ... imitated5.wav
#
# 🌊 Transformada: DWT sym4, nivel 2
# 🎧 Frecuencia: original, sin remuestreo
# ============================================================

In [2]:
# 📚 1. IMPORTAR LIBRERÍAS

import numpy as np
import soundfile as sf
import pywt

from pathlib import Path
from IPython.display import Audio, display, Markdown


In [3]:
# ============================================================
# ⚙️ 2. CONFIGURACIÓN
# ============================================================

TARGET_FILE = "audio.wav"
NUM_SECRETS = 5

WAVELET = "sym4"
LEVEL = 2

OUTPUT_DIR = Path("Imitation_Results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [4]:
# ============================================================
# 🎵 3. FUNCIÓN PARA LEER Y NORMALIZAR AUDIOS
# ============================================================

def load_audio(filename):
    """
    Lee un archivo WAV y normaliza su amplitud.

    Conserva:
       ✔ Frecuencia de muestreo original
       ✔ Número de muestras

    Requiere:
       ✔ Audio mono
    """

    signal, fs = sf.read(filename)

    # 🔎 Verificar que sea mono
    if signal.ndim != 1:
        raise ValueError(
            f"❌ {filename} no es mono."
        )

    # 📊 Normalización de amplitud
    max_amp = np.max(np.abs(signal))

    if max_amp == 0:
        raise ValueError(
            f"❌ {filename} contiene silencio."
        )

    signal = signal / max_amp

    return signal, fs


In [5]:
# ============================================================
# 🌊 4. FUNCIÓN IMITATION
# ============================================================

def imitation(source, target):
    """
    Genera una señal imitada y su clave de recuperación.

    Etapas:
       1️⃣ DWT de source y target.
       2️⃣ Ordenamiento de coeficientes.
       3️⃣ Reasignación de posiciones.
       4️⃣ Generación de la clave.
       5️⃣ Reconstrucción mediante IDWT.

    Retorna:
       imitated → Señal de audio imitada
       key      → Vector de permutación
    """

    # 🌊 DWT: descomposición nivel 2
    csource = pywt.wavedec(
        source, WAVELET, level=LEVEL
    )

    ctarget = pywt.wavedec(
        target, WAVELET, level=LEVEL
    )

    # 📊 Concatenar coeficientes [A2, D2, D1]
    cs = np.concatenate(csource)
    ct = np.concatenate(ctarget)

    if len(cs) != len(ct):
        raise ValueError(
            "❌ Los coeficientes tienen "
            "longitudes diferentes."
        )

    # 🔎 Ordenamiento de coeficientes
    index1 = np.argsort(cs)
    index2 = np.argsort(ct)

    # 🔄 Reasignar posiciones del secreto
    cs_m = np.zeros_like(cs)
    cs_m[index2] = cs[index1]

    # 🔑 Generar clave de recuperación
    key = np.zeros(len(cs), dtype=np.int64)
    key[index1] = index2

    # 📏 Longitudes de cada subbanda
    l1 = len(csource[0])
    l2 = len(csource[1])
    l3 = len(csource[2])

    # 🌊 Recuperar estructura wavelet
    csource_m = [
        cs_m[:l1],
        cs_m[l1:l1+l2],
        cs_m[l1+l2:l1+l2+l3]
    ]

    # 🔄 Reconstrucción IDWT
    imitated = pywt.waverec(
        csource_m, WAVELET
    )

    # Ajustar posibles muestras adicionales
    imitated = imitated[:len(source)]

    return imitated, key


In [6]:
# ============================================================
# 🎧 5. CARGAR AUDIO TARGET
# ============================================================

target, fs = load_audio(TARGET_FILE)

display(Markdown("## 🎵 Audio Target"))

print(f"📁 Archivo: {TARGET_FILE}")
print(f"🎧 Frecuencia: {fs} Hz")
print(f"⏱️ Duración: {len(target)/fs:.3f} s")
print(f"📊 Muestras: {len(target)}")

display(Audio(target, rate=fs))


## 🎵 Audio Target

📁 Archivo: audio.wav
🎧 Frecuencia: 44100 Hz
⏱️ Duración: 8.689 s
📊 Muestras: 383184


In [7]:
# ============================================================
# 🔐 6. GENERAR LAS CINCO CLAVES
# ============================================================

results = []

for i in range(1, NUM_SECRETS + 1):

    secret_file = f"secret{i}.wav"

    display(Markdown(
        f"### 🔐 Procesando mensaje secreto {i}"
    ))

    # 🎤 Leer secreto
    source, fs_secret = load_audio(secret_file)

    # 🔎 Verificar frecuencia de muestreo
    if fs_secret != fs:
        raise ValueError(
            f"❌ {secret_file}: frecuencia diferente."
        )

    # 🔎 Verificar número de muestras
    if len(source) != len(target):
        raise ValueError(
            f"❌ {secret_file}: longitud diferente."
        )

    # 🔄 Ejecutar Imitation
    imitated, key = imitation(source, target)

    # 📁 Archivos de salida
    key_file = OUTPUT_DIR / f"key{i}.csv"
    audio_file = OUTPUT_DIR / f"imitated{i}.wav"

    # 🔑 Guardar clave como índices enteros
    np.savetxt(
        key_file,
        key,
        fmt="%d",
        delimiter=","
    )

    # 💾 Guardar audio imitado
    sf.write(
        audio_file,
        imitated,
        fs,
        subtype="PCM_24"
    )

    # 📊 Registrar resultados
    results.append({
        "Secret": secret_file,
        "Key": str(key_file),
        "Imitated": str(audio_file),
        "Coefficients": len(key)
    })

    print(f"🎵 Secreto: {secret_file}")
    print(f"🔑 Clave: {key_file}")
    print(f"📊 Coeficientes: {len(key)}")
    print(f"💾 Audio imitado: {audio_file}")

    # 🎧 Reproducir señal imitada
    display(Audio(imitated, rate=fs))

    print("─" * 55)


### 🔐 Procesando mensaje secreto 1

🎵 Secreto: secret1.wav
🔑 Clave: Imitation_Results/key1.csv
📊 Coeficientes: 383197
💾 Audio imitado: Imitation_Results/imitated1.wav


───────────────────────────────────────────────────────


### 🔐 Procesando mensaje secreto 2

🎵 Secreto: secret2.wav
🔑 Clave: Imitation_Results/key2.csv
📊 Coeficientes: 383197
💾 Audio imitado: Imitation_Results/imitated2.wav


...

🎵 Secreto: secret4.wav
🔑 Clave: Imitation_Results/key4.csv
📊 Coeficientes: 383197
💾 Audio imitado: Imitation_Results/imitated4.wav


───────────────────────────────────────────────────────


### 🔐 Procesando mensaje secreto 5

🎵 Secreto: secret5.wav
🔑 Clave: Imitation_Results/key5.csv
📊 Coeficientes: 383197
💾 Audio imitado: Imitation_Results/imitated5.wav


───────────────────────────────────────────────────────


In [8]:
# ============================================================
# ✅ 7. RESUMEN FINAL
# ============================================================

display(Markdown(
    "## 🎉 ¡Generación de claves finalizada!"
))

print(f"🔑 Claves generadas: {len(results)}")
print(f"🎵 Frecuencia original: {fs} Hz")
print(f"🌊 Wavelet: {WAVELET}")
print(f"📊 Nivel DWT: {LEVEL}")
print(f"📁 Carpeta: {OUTPUT_DIR}")

for result in results:
    print(
        f"✅ {result['Key']} "
        f"({result['Coefficients']} índices)"
    )


## 🎉 ¡Generación de claves finalizada!

🔑 Claves generadas: 5
🎵 Frecuencia original: 44100 Hz
🌊 Wavelet: sym4
📊 Nivel DWT: 2
📁 Carpeta: Imitation_Results
✅ Imitation_Results/key1.csv (383197 índices)
✅ Imitation_Results/key2.csv (383197 índices)
✅ Imitation_Results/key3.csv (383197 índices)
✅ Imitation_Results/key4.csv (383197 índices)
✅ Imitation_Results/key5.csv (383197 índices)
